# 05 Exercise_Vector_Databases_Embeddings_RAG

# Industrial Robot Manual - RAG Agent Workflow

This notebook demonstrates a full **Retrieval-Augmented Generation (RAG)** pipeline:
1. **Environment Setup**: Installing `chromadb` and configuring the OpenAI client.
2. **Vector Database**: Initializing a Persistent ChromaDB client and a collection using OpenAI's `text-embedding-ada-002`.
3. **Data Ingestion**: Loading a technical manual, chunking the text with overlaps, and storing it in the vector database with line-number metadata.
4. **Tool Definition**: Creating a Pydantic-based tool schema for semantic searches.
5. **Agentic Interaction**: Using the OpenAI **Responses API** to simulate an agent that decides to call the `quote_lookup` tool to answer specific user technical questions based on the ingested manual.

In [1]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

### Create a collection

In [16]:
import chromadb
from chromadb import PersistentClient
from pathlib import Path

In [5]:
# Create a local folder where the db is located
chroma_client = PersistentClient('content/chromadb')
# Create a collection
chroma_collection = chroma_client.get_or_create_collection('manuals')

In [6]:
# Default embeddings for chromadb: onnx_models all-MiniLM-L6-v2
sample_result = chroma_collection.query(
    query_texts=["color red"],
    n_results=2
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 102MiB/s]


### Add a file to collection

In [17]:
# Add documents to the collection
PATH_TO_FILE = Path('sample_data/industrial_robot_manual.txt')
with PATH_TO_FILE.open() as f:
    manual_content = f.read()

# For simplicity, adding the entire manual as one document.
# In a real-world scenario, you might want to chunk this content.
chroma_collection.add(
    documents=[manual_content],
    metadatas=[{"source": "industrial_robot_manual.txt"}],
    ids=["industrial_robot_manual_1"]
)

In [13]:
# Inlcude embeddings to the results
sample_result = chroma_collection.query(
    query_texts=["Summary of the results"],
    # include=['embeddings'],
    n_results=2
)

sample_result

{'ids': [['industrial_robot_manual_1']],
 'embeddings': None,
 'documents': [['SECTION 1: SYSTEM ARCHITECTURE AND CORE SPECIFICATIONS\n1.1 Hardware Overview\nThe Aether-Net 9000 (Model: AN-9K) is an autonomous industrial logistics unit designed for high-density warehouse environments. The chassis is constructed from a proprietary aluminum-titanium alloy, designated as Ti-Al-44, which provides a 30% increase in structural rigidity compared to standard grade 2 titanium.\n\nDimensions: 1.2m x 0.8m x 1.1m\n\nNet Weight: 158.5 kg\n\nMaximum Payload Capacity: 450 kg\n\nProcessor: Dual-core Octa-Threaded Quantum-Ready Silicon (QRS-1)\n\n1.2 Drive System\nThe AN-9K utilizes a Quad-Directional Mecanum Wheel system, allowing for 360-degree holonomic movement. Each wheel is powered by an independent brushless DC motor (BLDC-400) capable of producing 45 Nm of torque. The maximum operational speed is governed by the firmware at 2.2 meters per second.\n\nSECTION 2: POWER MANAGEMENT AND BATTERY LOGIS

In [14]:
# Delete a document
chroma_collection.delete(
    ids=["industrial_robot_manual_1"]
)

{'deleted': 1}

### Add a file in chunks to collection

In [46]:
# Add documents to the collection
PATH_TO_FILE = Path('sample_data/industrial_robot_manual.txt')
with PATH_TO_FILE.open() as f:
    f_lines = f.readlines()

f_lines = [fl.strip() for fl in f_lines]
f_lines = [fl.strip() for fl in f_lines if fl != '']

# Simple chunking strategy
chunk_size  = 5
overlap_size = 2

chunks = []
idx = 0
while idx < len(f_lines):
  start= idx
  end=idx + chunk_size

  # print(start, end)

  current = '\n'.join(f_lines[start:end])
  chunks.append(current)

  idx = end - overlap_size

In [42]:
# Check
f_lines[0:5]
chunks[0]

f_lines[3:8]
chunks[1]

['SECTION 1: SYSTEM ARCHITECTURE AND CORE SPECIFICATIONS',
 '1.1 Hardware Overview',
 'The Aether-Net 9000 (Model: AN-9K) is an autonomous industrial logistics unit designed for high-density warehouse environments. The chassis is constructed from a proprietary aluminum-titanium alloy, designated as Ti-Al-44, which provides a 30% increase in structural rigidity compared to standard grade 2 titanium.',
 'Dimensions: 1.2m x 0.8m x 1.1m',
 'Net Weight: 158.5 kg']

'Dimensions: 1.2m x 0.8m x 1.1m\nNet Weight: 158.5 kg\nMaximum Payload Capacity: 450 kg\nProcessor: Dual-core Octa-Threaded Quantum-Ready Silicon (QRS-1)\n1.2 Drive System'

In [50]:
chunk_idx = 0
batch_size = 100

while chunk_idx < len(chunks):
  start_batch = chunk_idx
  end_batch = min(chunk_idx + batch_size, len(chunks)) # Ensure end_batch does not exceed chunks length
  current_chunk = chunks[start_batch:end_batch]

  batch_metadatas = []
  batch_ids = []

  for i, doc_content in enumerate(current_chunk):
    global_chunk_index = start_batch + i

    start_line_approx = global_chunk_index * (chunk_size - overlap_size)
    end_line_approx = start_line_approx + chunk_size - 1

    batch_metadatas.append({
        "source": "industrial_robot_manual.txt",
        "chunk_index": global_chunk_index,
        "start_line_approx": start_line_approx,
        "end_line_approx": end_line_approx
    })
    batch_ids.append(f"industrial_robot_manual_chunk_{global_chunk_index}_lines_{start_line_approx}-{end_line_approx}")

  chroma_collection.add(
    documents=current_chunk,
    metadatas=batch_metadatas,
    ids=batch_ids
  )

  chunk_idx = end_batch

In [53]:
chroma_collection

Collection(name=manuals)

In [54]:
sample_result = chroma_collection.query(
    query_texts=["Whatd does code red mean"],
    # include=['embeddings'],
    n_results=2
)

sample_result

{'ids': [['industrial_robot_manual_chunk_9_lines_27-31',
   'industrial_robot_manual_chunk_10_lines_30-34']],
 'embeddings': None,
 'documents': [['When the robot encounters an object, it assigns a priority level:\nCode GREEN: Stationary object, static map update.\nCode YELLOW: Moving object (human/other robot), predictive pathing engaged.\nCode RED: Immediate collision risk, 0.2s emergency braking.\nSECTION 4: ERROR CODES AND TROUBLESHOOTING',
   'Code RED: Immediate collision risk, 0.2s emergency braking.\nSECTION 4: ERROR CODES AND TROUBLESHOOTING\n4.1 Fatal System Errors\nIf a fatal error occurs, the LED ring on the robot will pulse Red and display a code on the auxiliary OLED screen.\nError 0x001: IMU Calibration Mismatch. Solution: Perform a "Static Reset" on level ground.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'chunk_index': 9,
    'start_line_approx': 27,
    'end_line_approx': 31,
    'source': 'industrial_robot

In [67]:
sample_result['documents']

[['When the robot encounters an object, it assigns a priority level:\nCode GREEN: Stationary object, static map update.\nCode YELLOW: Moving object (human/other robot), predictive pathing engaged.\nCode RED: Immediate collision risk, 0.2s emergency braking.\nSECTION 4: ERROR CODES AND TROUBLESHOOTING',
  'Code RED: Immediate collision risk, 0.2s emergency braking.\nSECTION 4: ERROR CODES AND TROUBLESHOOTING\n4.1 Fatal System Errors\nIf a fatal error occurs, the LED ring on the robot will pulse Red and display a code on the auxiliary OLED screen.\nError 0x001: IMU Calibration Mismatch. Solution: Perform a "Static Reset" on level ground.']]

# Craete a new collection using OpenAI emebeddings

In [70]:
from google.colab import userdata
from openai import OpenAI

from chromadb.utils import embedding_functions


In [71]:
OpenAIEmbeddings = embedding_functions.OpenAIEmbeddingFunction(
    api_key=userdata.get('open-ai-key'),
    model_name="text-embedding-ada-002"
)

chroma_collection = chromadb.PersistentClient('content/chromadb').get_or_create_collection(
    name='manuals_openai',
    embedding_function=OpenAIEmbeddings)


In [73]:
sample_result = chroma_collection.query(
    query_texts=["Whatd does code red mean"],
    n_results=2
)

sample_result

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [74]:
chunk_idx = 0
batch_size = 100

while chunk_idx < len(chunks):
  start_batch = chunk_idx
  end_batch = min(chunk_idx + batch_size, len(chunks)) # Ensure end_batch does not exceed chunks length
  current_chunk = chunks[start_batch:end_batch]

  batch_metadatas = []
  batch_ids = []

  for i, doc_content in enumerate(current_chunk):
    global_chunk_index = start_batch + i

    start_line_approx = global_chunk_index * (chunk_size - overlap_size)
    end_line_approx = start_line_approx + chunk_size - 1

    batch_metadatas.append({
        "source": "industrial_robot_manual.txt",
        "chunk_index": global_chunk_index,
        "start_line_approx": start_line_approx,
        "end_line_approx": end_line_approx
    })
    batch_ids.append(f"industrial_robot_manual_chunk_{global_chunk_index}_lines_{start_line_approx}-{end_line_approx}")

  chroma_collection.add(
    documents=current_chunk,
    metadatas=batch_metadatas,
    ids=batch_ids
  )

  chunk_idx = end_batch

In [76]:
sample_result = chroma_collection.query(
    query_texts=["What does code green mean"],
    n_results=2
)

sample_result

{'ids': [['industrial_robot_manual_chunk_9_lines_27-31',
   'industrial_robot_manual_chunk_11_lines_33-37']],
 'embeddings': None,
 'documents': [['When the robot encounters an object, it assigns a priority level:\nCode GREEN: Stationary object, static map update.\nCode YELLOW: Moving object (human/other robot), predictive pathing engaged.\nCode RED: Immediate collision risk, 0.2s emergency braking.\nSECTION 4: ERROR CODES AND TROUBLESHOOTING',
   'If a fatal error occurs, the LED ring on the robot will pulse Red and display a code on the auxiliary OLED screen.\nError 0x001: IMU Calibration Mismatch. Solution: Perform a "Static Reset" on level ground.\nError 0x044: Thermal Overload in Motor Controller 3. Solution: Inspect for tangled debris in the rear-left wheel.\nError 0x999: Encryption Handshake Failure. Solution: Re-sync the security dongle with the Ground Control Station.\n4.2 Maintenance Log Codes']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': N

# Building an agent

In [78]:
from pydantic import BaseModel

In [79]:
# (args) -> Quote

class QuoteLookupArgs(BaseModel):
  query:str
  top_n: int

In [81]:
QuoteLookupArgs.model_json_schema()

{'properties': {'query': {'title': 'Query', 'type': 'string'},
  'top_n': {'title': 'Top N', 'type': 'integer'}},
 'required': ['query', 'top_n'],
 'title': 'QuoteLookupArgs',
 'type': 'object'}

In [102]:

open_ai_key = userdata.get("open-ai-key")
if not open_ai_key:
    raise RuntimeError("Missing OpenAI API key: 'open-ai-key'")

o_client = OpenAI(api_key=open_ai_key)


In [103]:

schema = QuoteLookupArgs.model_json_schema()
schema["additionalProperties"] = False

# Responses API format: name, description, parameters, and strict
TOOLS = [
    {
        "type": "function",
        "name": "quote_lookup",
        "description": (
            "Look up a quote from the internal collection of manuals "
            "that corresponds semantically to a natural-language query."
        ),
        "parameters": schema,
        "strict": True,
    }
]


In [104]:

# Add the tool in our conversation
response = o_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for manual analysis and quote "
                "finding. Use quote_lookup to answer the user's question."
            ),
        },
        {
            "role": "user",
            "content": "What does code green mean?",
        },
    ],
    tools=TOOLS,
    tool_choice="auto",
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"code green","top_n":5}', call_id='call_NudcGd6UfiXTCGRo8kGB32HR', name='quote_lookup', type='function_call', id='fc_058a7f3a954f031f006a65c6a174d8819584c364f1569e6de3', caller=None, namespace=None, status='completed')]


In [113]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"code green","top_n":5}', call_id='call_NudcGd6UfiXTCGRo8kGB32HR', name='quote_lookup', type='function_call', id='fc_058a7f3a954f031f006a65c6a174d8819584c364f1569e6de3', caller=None, namespace=None, status='completed')]

# Get the results from the tool_calls
* The main idea is to get the results /text output from the tool calls and feed them in a OpenAI call and the result would be an answer based on out own vector database

In [128]:
# import json
# # Get all tool calls
# tool_calls =  [x for x in response.output if x.type == 'function_call']
# TOOL_HANDLERS = {
#     "quote_lookup": lambda args: chroma_collection.query(
#         query_texts=[args.query],
#         n_results=args.top_n
#     )
# }

# tool_calls[0]

# # Iterate through tool calls and execute handlers
# results = []
# for rc in tool_calls:
#     if rc.name in TOOL_HANDLERS:
#         # Parse arguments into the Pydantic model
#         args_data = json.loads(rc.arguments)
#         args_obj = QuoteLookupArgs(**args_data)

#         # Execute the handler
#         query_result = TOOL_HANDLERS[rc.name](args_obj)
#         results.append(query_result)

# # Display the retrieved documents from ChromaDB
# for i, res in enumerate(results):
#     print(f"Result for tool call {i}:")
#     display(res['documents'])


In [126]:
tool_calls = [
    item
    for item in response.output
    if item.type == "function_call"
]

if len(tool_calls) != len(results):
    raise ValueError(
        f"Expected {len(tool_calls)} tool results, got {len(results)}"
    )

tool_outputs = []

for call, result in zip(tool_calls, results):
    tool_outputs.append(
        {
            "type": "function_call_output",
            "call_id": call.call_id,
            # The Responses API expects the output to normally be a string.
            "output": json.dumps(result["documents"]),
        }
    )
tool_outputs

[{'type': 'function_call_output',
  'call_id': 'call_NudcGd6UfiXTCGRo8kGB32HR',
  'output': '[["When the robot encounters an object, it assigns a priority level:\\nCode GREEN: Stationary object, static map update.\\nCode YELLOW: Moving object (human/other robot), predictive pathing engaged.\\nCode RED: Immediate collision risk, 0.2s emergency braking.\\nSECTION 4: ERROR CODES AND TROUBLESHOOTING", "Code RED: Immediate collision risk, 0.2s emergency braking.\\nSECTION 4: ERROR CODES AND TROUBLESHOOTING\\n4.1 Fatal System Errors\\nIf a fatal error occurs, the LED ring on the robot will pulse Red and display a code on the auxiliary OLED screen.\\nError 0x001: IMU Calibration Mismatch. Solution: Perform a \\"Static Reset\\" on level ground.", "If a fatal error occurs, the LED ring on the robot will pulse Red and display a code on the auxiliary OLED screen.\\nError 0x001: IMU Calibration Mismatch. Solution: Perform a \\"Static Reset\\" on level ground.\\nError 0x044: Thermal Overload in Mot

In [127]:

final_response = o_client.responses.create(
    model="gpt-4o-mini",

    # Includes the original user message and model function calls.
    previous_response_id=response.id,

    # Supply only the results of those function calls.
    input=tool_outputs,

    # Instructions are not automatically carried forward when using
    # previous_response_id, so provide them again when needed.
    instructions=(
        "You are a helpful assistant for manual analysis and quote finding. "
        "Answer the user's question using the retrieved manual excerpts."
    ),
)

print(final_response.output_text)

"Code GREEN" refers to a stationary object encountered by a robot, indicating that a static map update is necessary. This code signals that there is no immediate need for the robot to take evasive action since the object is not moving.


# RAG conversation

In [139]:
import json
from openai import OpenAI

client = OpenAI(api_key=userdata.get("open-ai-key"))

# 1. Real search logic using ChromaDB

def quote_lookup(query, top_n=5):
    """Search the manuals_openai collection for relevant quotes."""
    # Use the chroma_collection object defined in previous cells
    results = chroma_collection.query(
        query_texts=[query],
        n_results=top_n
    )
    return results


# 2. Map tool names to Python functions

TOOL_FUNCTIONS = {
    "quote_lookup": quote_lookup,
}


# 3. Describe tools to OpenAI
TOOLS = [
    {
        "type": "function",
        "name": "quote_lookup",
        "description": "Search manuals for a relevant quote.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The question to search for.",
                },
                "top_n": {
                    "type": "integer",
                    "description": "Number of results to return.",
                }
            },
            "required": ["query", "top_n"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

In [143]:
SYSTEM_PROMPT = """
You answer questions about manuals.

Use the available tools when information must be retrieved.
Base your answer on the tool results.
""".strip()


def ask(message, previous_response_id=None):
    request = {
        "model": "gpt-4o-mini",
        "instructions": SYSTEM_PROMPT,
        "input": message,
        "tools": TOOLS,
    }

    if previous_response_id:
        request["previous_response_id"] = previous_response_id

    response = client.responses.create(**request)

    # Keep going until the model returns a normal text answer.
    while True:
        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        if not tool_calls:
            return response.output_text, response.id

        tool_results = []

        for call in tool_calls:
            # Logging tool usage
            print(f"---> [Tool Call] Executing: {call.name} with args: {call.arguments}")

            function = TOOL_FUNCTIONS[call.name]
            arguments = json.loads(call.arguments)

            result = function(**arguments)

            tool_results.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

        response = client.responses.create(
            model="gpt-4o-mini",
            instructions=SYSTEM_PROMPT,
            previous_response_id=response.id,
            input=tool_results,
            tools=TOOLS,
        )

In [144]:
previous_id = None

answer, previous_id = ask(
    "What does code green mean?",
    previous_id,
)
print("Assistant:", answer)

answer, previous_id = ask(
    "Which manual and page did that come from?",
    previous_id,
)
print("Assistant:", answer)

answer, previous_id = ask(
    "Explain it more simply.",
    previous_id,
)
print("Assistant:", answer)

---> [Tool Call] Executing: quote_lookup with args: {"query":"code green","top_n":5}
Assistant: In the context of a robotic system, "Code Green" refers to a scenario where the robot encounters a **stationary object**, which triggers a **static map update**. This indicates that the object is not moving and can be safely logged into the robot's environmental understanding without immediate risk. 

Here’s a brief overview of related codes:
- **Code Yellow**: For moving objects (like humans or other robots), predictive pathing is engaged.
- **Code Red**: Indicates an immediate collision risk, prompting emergency braking within 0.2 seconds.
Assistant: The information about "Code Green" comes from the **industrial robot manual**. The relevant details can be found in the following sections:

- **Source**: Industrial Robot Manual
- **Page/Section**: Approximately from lines 27 to 31 in the manual.
Assistant: "Code Green" means the robot sees a **stationary object** in its path. It notes this s